<a href="https://colab.research.google.com/github/dhruvbadhe/nova-task-1/blob/main/WaterPotabillty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/water_potability_synthetic.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.corr()

# MID MULTICOLLINEARITY

In [ ]:
columns = df.columns

In [ ]:
for col in columns:
 sns.boxplot(df[col])
 plt.show()

HAS MISSING VALUES - KNN, MEDIAN

HAS OUTLIERS

DATA SCALING FOR MISSING VALUES

FEATURE TRANSFORMATION - LOG

FEATURE SELECTION

PIPELINES

GRID SEARCH OR OPTUNA

In [ ]:
x =df.iloc[:,:-1]

In [ ]:
x

In [ ]:
y = df.iloc[:,-1]

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train = pd.DataFrame(x_train, columns=x.columns)
X_test = pd.DataFrame(x_test, columns=x.columns)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [ ]:
pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"scale_pos_weight used: {pos_weight:.2f}")

In [ ]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=pos_weight,
        verbosity=0
    ))
])


In [ ]:
param_grid = {
    'classifier__n_estimators': [400, 600, 800],
    'classifier__max_depth': [6, 7, 8, 9],
    'classifier__learning_rate': [0.03, 0.05, 0.07],
    'classifier__subsample': [0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.7, 0.8, 0.9],
    'classifier__min_child_weight': [1, 3, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [ ]:
pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

In [ ]:
random_search.fit(X_train, y_train)


best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)


print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f} ({accuracy_score(y_test, y_pred)*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))